In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("/app/notebooks/data/synthetic/10000p/")
KEYS = ["SUBJECT", "SPELL_IDENTIFIER"]

def keep_first_spell(df: pd.DataFrame, name: str = "episodes") -> pd.DataFrame:
    """Keep the first source row for each SUBJECT/SPELL_IDENTIFIER pair."""
    missing = [c for c in KEYS if c not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing key columns: {missing}")
    result = df.copy()
    before = len(result)
    result = result.drop_duplicates(KEYS, keep="first").reset_index(drop=True)
    print(f"{name}: {before:,} rows -> {len(result):,} unique spells")
    return result

print(f"Using data from: {DATA_DIR}")


Using data from: /app/notebooks/data/synthetic/10000p


In [14]:
df_episodes = pd.read_csv(
    DATA_DIR / "icare_episodes_anon.csv",
    parse_dates=["ADMISSION_DATE", "DISCHARGE_DATE"],
)
df_episodes = keep_first_spell(df_episodes)

episodes: 10,000 rows -> 10,000 unique spells


In [15]:
# ------------------------------------------------
# Compute history of events (first, last, count)
# ------------------------------------------------
# Libraries
from icare_risk.clinphen.utils import compute_phenotype_history

# 1. Load the RAW events (not the aggregated wide matrix)
df_events_raw = pd.read_parquet(DATA_DIR / '02a_derived_events_extracted_raw.parquet')

# 2. Include the index_admission column
df_episodes["index_admission"] = pd.to_datetime(
    df_episodes["ADMISSION_DATE"].astype(str) + " "
    + df_episodes["ADMISSION_TIME"].astype(str),
    errors="coerce",
)

# Print the occurrence counts of extracted events
event_counts = df_events_raw["event_name"].value_counts()
print(f"--- Extracted {len(event_counts)} Unique Event Components ---\n")
print(event_counts)

# 2. Compute recency for specific phenotypes (or leave None for all)
df = compute_phenotype_history(
    df_episodes=df_episodes,
    df_events_raw=df_events_raw,
    target_phenotypes=None,
    metrics=None,  # e.g., ["last", "first", "count"]
)

# 3. Save file
filename = '02b_derived_features_phenotypes.parquet'
df.to_parquet(DATA_DIR / filename)
print(f"\nFile successfully saved in: {DATA_DIR / filename}")

--- Extracted 33 Unique Event Components ---

event_name
gavaghan_hx_prior_antibiotics_90d__comp1    20102
jones_hx_prior_antibiotics_30d__comp1       20102
kim_hx_recent_antibiotics_90d__comp1        20102
tumbarello_hx_recent_abx_90d__comp0         20102
kim_hx_prior_antibiotics_90d__comp0         20102
charlson_hx_diabetes_uncomp__comp2          19884
jones_hx_prior_antibiotics_30d__comp0       10098
kim_hx_recent_antibiotics_90d__comp0        10098
charlson_hx_diabetes_uncomp__comp0            828
charlson_hx_renal_mod_sev__comp0              240
charlson_hx_pulmonary__comp0                  175
charlson_hx_dementia__comp0                   145
charlson_hx_stroke__comp0                      99
charlson_hx_mi__comp0                          94
charlson_hx_pud__comp0                         94
charlson_hx_chf__comp0                         93
charlson_hx_rheum__comp0                       86
charlson_hx_liver_mod_sev__comp0               48
kim_hx_prior_esbl__comp0                   

In [20]:
from icare_risk.clinphen.utils.debug import (
    print_phenotype_column_matrix,
    print_phenotype_columns_by_metric,
    summarize_phenotype_history
)

# Option A: See a compact per-phenotype matrix view
print_phenotype_column_matrix(df)

# Option B: See a flat, categorized list grouped by metric type
print_phenotype_columns_by_metric(df)

# Option C: Full distribution summary stats and report
df_report = summarize_phenotype_history(
    df, print_report=True, output_csv_path=None
)


--- Phenotype Column Matrix (29 phenotypes found) ---

  charlson_hx_aids                  | [✓] last   [✓] first   [✓] count
  charlson_hx_cancer_met            | [✓] last   [✓] first   [✓] count
  charlson_hx_cancer_solid          | [✓] last   [✓] first   [✓] count
  charlson_hx_chf                   | [✓] last   [✓] first   [✓] count
  charlson_hx_dementia              | [✓] last   [✓] first   [✓] count
  charlson_hx_diabetes_comp         | [✓] last   [✓] first   [✓] count
  charlson_hx_diabetes_uncomp       | [✓] last   [✓] first   [✓] count
  charlson_hx_hiv                   | [✓] last   [✓] first   [✓] count
  charlson_hx_leukemia              | [✓] last   [✓] first   [✓] count
  charlson_hx_liver_mild            | [✓] last   [✓] first   [✓] count
  charlson_hx_liver_mod_sev         | [✓] last   [✓] first   [✓] count
  charlson_hx_lymphoma              | [✓] last   [✓] first   [✓] count
  charlson_hx_mi                    | [✓] last   [✓] first   [✓] count
  charlson_hx_pud    

In [12]:
print(df.columns.tolist())

['SUBJECT', 'SPELL_IDENTIFIER', 'ENCNTR_ID', 'EPISODE_IDENTIFIER', 'ADMISSION_DATE', 'ADMISSION_TIME', 'AGE_AT_ADMISSION', 'DISCHARGE_DATE', 'MAIN_SPECIALTY_CODE', 'MAIN_SPECIALTY_CODE_DESC', 'INDEX_OF_MULTIPLE_DEPRIVATION_DECILE', 'index_admission', 'days_since_charlson_hx_mi_last', 'days_since_charlson_hx_mi_first', 'count_prior_charlson_hx_mi', 'days_since_charlson_hx_chf_last', 'days_since_charlson_hx_chf_first', 'count_prior_charlson_hx_chf', 'days_since_charlson_hx_pvd_last', 'days_since_charlson_hx_pvd_first', 'count_prior_charlson_hx_pvd', 'days_since_charlson_hx_stroke_last', 'days_since_charlson_hx_stroke_first', 'count_prior_charlson_hx_stroke', 'days_since_charlson_hx_dementia_last', 'days_since_charlson_hx_dementia_first', 'count_prior_charlson_hx_dementia', 'days_since_charlson_hx_pulmonary_last', 'days_since_charlson_hx_pulmonary_first', 'count_prior_charlson_hx_pulmonary', 'days_since_charlson_hx_rheum_last', 'days_since_charlson_hx_rheum_first', 'count_prior_charlson_h